In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date

from prophet import Prophet
from prophet.diagnostics import performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.diagnostics import cross_validation

from utils import extrair_feriados_prophet, calcular_metricas_completas, executar_pipeline_otimizacao

c:\Users\Windows 10\Desktop\rosman-store-project\sales-forecast-with-prophet\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [26]:
df = pd.read_csv("../data/resultados_otimizacao_legado.csv", sep=",", on_bad_lines="skip")
df_01 = pd.read_csv("../data/resultados_otimizacao.csv", sep=",")

In [18]:
df[df['metodo'] == 'Holdout'].head(50)

,data_execucao,metodo,changepoint_prior_scale,seasonality_mode,seasonality_prior_scale,MSE,RMSE,MAE,MAPE,MDAPE,SMAPE,tempo_execucao
0,2026-07-07 23:52:04,Holdout,0.01,additive,0.1,160088.309796,400.110372,314.151595,0.066333,0.052109,0.324879,1.81
1,2026-07-07 23:52:05,Holdout,0.01,additive,1.0,166214.611283,407.694262,321.612264,0.067587,0.052444,0.326208,0.43
2,2026-07-07 23:52:06,Holdout,0.01,additive,10.0,166341.027123,407.849270,321.155317,0.067395,0.052172,0.326064,0.48
3,2026-07-07 23:52:07,Holdout,0.01,multiplicative,0.1,138198.962283,371.751210,276.846381,0.064660,0.043085,0.323355,0.51
4,2026-07-07 23:52:08,Holdout,0.01,multiplicative,1.0,139812.060253,373.914509,279.913732,0.065306,0.043853,0.323957,0.40
5,2026-07-07 23:52:09,Holdout,0.01,multiplicative,10.0,136748.024646,369.794571,277.004094,0.064243,0.044307,0.322970,0.37
6,2026-07-07 23:52:10,Holdout,0.10,additive,0.1,141378.621693,376.003486,293.285296,0.064125,0.055675,0.321966,0.43
7,2026-07-07 23:52:11,Holdout,0.10,additive,1.0,141457.725123,376.108661,293.309841,0.064127,0.055678,0.321962,0.46
8,2026-07-07 23:52:12,Holdout,0.10,additive,10.0,141573.798449,376.262938,293.458502,0.064127,0.055618,0.321974,0.46
9,2026-07-07 23:52:13,Holdout,0.10,multiplicative,0.1,142077.403348,376.931563,274.975554,0.065894,0.043974,0.324765,0.45


In [25]:
df[df['metodo'] == 'Validacao Cruzada'].head(100)

,data_execucao,metodo,changepoint_prior_scale,seasonality_mode,seasonality_prior_scale,MSE,RMSE,MAE,MAPE,MDAPE,SMAPE,tempo_execucao
18,2026-07-08 00:03:53,Validacao Cruzada,0.01,additive,0.1,714.225906,542.503925,NaN,0.125096,0.757516,18.17,NaN
19,2026-07-08 00:04:04,Validacao Cruzada,0.01,additive,1.0,708.720406,537.227315,NaN,0.123456,0.765359,10.44,NaN
20,2026-07-08 00:04:17,Validacao Cruzada,0.01,additive,10.0,708.205733,536.478640,NaN,0.123853,0.756209,12.69,NaN
21,2026-07-08 00:04:38,Validacao Cruzada,0.01,multiplicative,0.1,713.618516,526.289397,NaN,0.126868,0.758824,20.24,NaN
22,2026-07-08 00:04:51,Validacao Cruzada,0.01,multiplicative,1.0,717.123801,530.006496,NaN,0.126527,0.758824,11.94,NaN
23,2026-07-08 00:05:02,Validacao Cruzada,0.01,multiplicative,10.0,707.700605,521.017705,NaN,0.124736,0.766667,10.59,NaN
24,2026-07-08 00:05:13,Validacao Cruzada,0.10,additive,0.1,726.645058,552.862188,NaN,0.134552,0.736601,11.10,NaN
25,2026-07-08 00:05:26,Validacao Cruzada,0.10,additive,1.0,732.680895,556.619673,NaN,0.134263,0.729412,11.86,NaN
26,2026-07-08 00:05:37,Validacao Cruzada,0.10,additive,10.0,730.273247,555.687396,NaN,0.133492,0.733333,11.34,NaN
27,2026-07-08 00:05:50,Validacao Cruzada,0.10,multiplicative,0.1,858.044923,624.888739,NaN,0.152120,0.694771,12.33,NaN


In [27]:
df_01[df_01['metodo'] == 'Validacao Cruzada'].head(100)

,data_execucao,metodo,parametros,rmse,mae,mape,mdape,coverage,tempo_execucao
0,2026-07-08 00:40:53,Validacao Cruzada,"{'changepoint_prior_scale': 0.01, 'daily_seaso...",705.992189,534.443161,NaN,0.123806,0.764052,9.10
1,2026-07-08 00:41:02,Validacao Cruzada,"{'changepoint_prior_scale': 0.01, 'daily_seaso...",717.614532,544.556200,NaN,0.125307,0.752941,9.37
2,2026-07-08 00:41:12,Validacao Cruzada,"{'changepoint_prior_scale': 0.01, 'daily_seaso...",718.326890,544.951416,NaN,0.125318,0.754902,8.92
3,2026-07-08 00:41:21,Validacao Cruzada,"{'changepoint_prior_scale': 0.01, 'daily_seaso...",705.264591,519.453781,NaN,0.124818,0.769935,8.87
4,2026-07-08 00:41:30,Validacao Cruzada,"{'changepoint_prior_scale': 0.01, 'daily_seaso...",714.310621,525.297424,NaN,0.125662,0.767974,8.80
5,2026-07-08 00:41:40,Validacao Cruzada,"{'changepoint_prior_scale': 0.01, 'daily_seaso...",707.802467,520.508675,NaN,0.122967,0.771242,9.38
6,2026-07-08 00:41:51,Validacao Cruzada,"{'changepoint_prior_scale': 0.1, 'daily_season...",730.392800,555.521836,NaN,0.134157,0.728758,10.84
7,2026-07-08 00:42:01,Validacao Cruzada,"{'changepoint_prior_scale': 0.1, 'daily_season...",729.555011,554.940878,NaN,0.134185,0.732680,9.59
8,2026-07-08 00:42:12,Validacao Cruzada,"{'changepoint_prior_scale': 0.1, 'daily_season...",729.858437,555.136031,NaN,0.133926,0.733333,9.86
9,2026-07-08 00:42:22,Validacao Cruzada,"{'changepoint_prior_scale': 0.1, 'daily_season...",896.200601,649.010440,NaN,0.158693,0.675163,9.57
